In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Load your single CSV file
df = pd.read_csv('NASA Turbofan Engine.csv')

# Drop non-sensor columns ('unit', 'cycle', and operational settings) 
# keeping only the sensor measurements for training/evaluation
features = df.drop(['unit', 'cycle', 'setting1', 'setting2', 'setting3'], axis=1)

# For unsupervised training, we use the healthy engine readings (e.g., early cycles)
# or all rows if they represent standard engine operations.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(features)

# Helper function to create time-series windows: shape becomes (samples, timesteps, features)
def create_windows(data, timesteps=10):
    windows = []
    for i in range(len(data) - timesteps):
        windows.append(data[i:(i + timesteps)])
    return np.array(windows)

TIMESTEPS = 10
windows = create_windows(scaled_data, timesteps=TIMESTEPS)

# Split windows into train and validation sets (strictly on normal/healthy data)
X_train, X_test_normal = train_test_split(windows, test_size=0.2, random_state=42)

n_features = X_train.shape[2]

# 2. LSTM Autoencoder Architecture
model = keras.Sequential([
    # Encoder
    layers.Input(shape=(TIMESTEPS, n_features)),
    layers.LSTM(32, activation='relu', return_sequences=False),
    
    # Bridge across timesteps
    layers.RepeatVector(TIMESTEPS),
    
    # Decoder
    layers.LSTM(32, activation='relu', return_sequences=True),
    layers.TimeDistributed(layers.Dense(n_features))
])

model.compile(optimizer='adam', loss='mse')

# 3. Train strictly on normal sequential sensor data
H = model.fit(X_train, X_train, epochs=10, batch_size=32, validation_data=(X_test_normal, X_test_normal))

# 4. Set Threshold using normal validation windows
X_test_preds = model.predict(X_test_normal)
normal_mse = np.mean(np.power(X_test_normal - X_test_preds, 2), axis=(1, 2))
threshold = np.percentile(normal_mse, 99)

# 5. Evaluate on test windows to detect anomalies/faults
test_preds = model.predict(windows)
test_mse = np.mean(np.power(windows - test_preds, 2), axis=(1, 2))

faults = test_mse > threshold
print(f"Total Detected Engine Fault Instances: {sum(faults)}")